# Validar a EDA do TCC no Google Colab

Abra o notebook no Colab. No menu **Ambiente de execução → Alterar o tipo de ambiente de execução**, selecione **CPU** (sem acelerador) e a versão **2026.07**, com Python 3.12. [Versões do Colab](https://research.google.com/colaboratory/runtime-version-faq.html)

Execute as quatro células em ordem. A célula 1 usa as mesmas versões de **00_VALIDAR_AMBIENTE** e reaproveita o ambiente se ele estiver disponível na mesma sessão. Em uma nova sessão, a instalação ocorre em outro ambiente temporário. A EDA roda na cópia criada pelo executor em **validation_runs/**. Esta etapa inclui os diagnósticos exploratórios do Meridian e termina antes da amostragem posterior.

São esperadas duas falhas de hash da introdução já conhecidas no commit auditado. O notebook registra essas falhas. Se a célula 2 ou 3 parar com erro, vá direto à **célula 4** para baixar o diagnóstico parcial.

No resultado esperado, as **129 tabelas históricas e os 15 CSVs temporais** concordam. Envie ao Codex o ZIP baixado na célula 4, inclusive quando houver divergências.


## 1. Preparar o ambiente e a cópia de referência

In [ ]:
import platform
import json
import os
from hashlib import sha256
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

requirements_verified = False
integrity_ok = False
run_dir = None
run_info = None

if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Abra este notebook em um runtime Linux do Google Colab.")

try:
    from google import colab
except ImportError as error:
    raise RuntimeError("Abra este notebook no Google Colab.") from error

print("Python do Colab", platform.python_version(), flush=True)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Selecione o runtime Colab 2026.07 com Python 3.12 e rode esta célula novamente.")

git = shutil.which("git")
if git is None:
    raise RuntimeError("O executável do Git não foi encontrado no runtime.")

REPO_URL = "https://github.com/thiagossilva17/TCC_MMM_Google_Meridian.git"
REPO_COMMIT = "007f487d9f541c34a43395412c9637254bd65c04"
MERIDIAN_URL = "https://github.com/google/meridian.git"
MERIDIAN_COMMIT = "02111531f8661373aa7b6ba31c316c67d72d1dd2"
MERIDIAN_VERSION = "2.1.0"

work_dir = Path(tempfile.mkdtemp(prefix="tcc-eda-", dir="/content"))
checkout = work_dir / "repositorio"
venv_dir = work_dir / "venv"

subprocess.run([git, "clone", "--quiet", "--no-checkout", "--no-tags", REPO_URL, str(checkout)], check=True)
subprocess.run([git, "checkout", "--quiet", "--detach", REPO_COMMIT], cwd=checkout, check=True)
actual_commit = subprocess.check_output([git, "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
if actual_commit != REPO_COMMIT:
    raise RuntimeError(f"Commit do TCC inesperado ({actual_commit}).")

print("Commit do TCC confirmado", actual_commit)

verify_installation = r'''
import importlib.metadata as metadata
import json
import sys
from pathlib import Path

from packaging.requirements import Requirement

expected_url, expected_commit, expected_version = sys.argv[1:]
if sys.version_info[:2] != (3, 12) or sys.prefix == sys.base_prefix:
    raise RuntimeError("O interpretador isolado esperado é Python 3.12.")

for filename in ("requirements-temporal.txt", "constraints-tested.txt"):
    verified = 0
    for line in Path(filename).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        requirement = Requirement(line)
        if requirement.marker and not requirement.marker.evaluate():
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            if filename == "constraints-tested.txt":
                continue
            raise
        if not requirement.specifier.contains(installed, prereleases=True):
            raise RuntimeError(f"{requirement} é exigido. Versão instalada de {requirement.name}: {installed}.")
        verified += 1
    print(f"{filename} — {verified} versões conferidas.")

meridian = metadata.distribution("google-meridian")
origin = json.loads(meridian.read_text("direct_url.json") or "{}")
if meridian.version != expected_version:
    raise RuntimeError(f"Versão inesperada do Meridian ({meridian.version}).")
if origin.get("url") != expected_url or origin.get("vcs_info", {}).get("commit_id") != expected_commit:
    raise RuntimeError("A origem do Meridian instalado diverge do commit fixado.")

print("Versão e commit do Meridian confirmados", meridian.version, expected_commit)
'''


report_dir = work_dir / "relatorios"
report_dir.mkdir()


def write_report(filename, content):
    (report_dir / filename).write_text(json.dumps(content, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def digest(path):
    return sha256(path.read_bytes()).hexdigest()

def python_command(environment):
    return [str(environment / "bin" / "python"), "-I", "-B", "-X", "utf8", "-u"]

def pip_command(environment):
    return [sys.executable, "-I", "-m", "pip", "--isolated", "--disable-pip-version-check", "--python", str(environment)]

def verify_environment(environment):
    dependencies = subprocess.run([*pip_command(environment), "check"], cwd=checkout)
    if dependencies.returncode:
        return False
    versions = subprocess.run(
        [*python_command(environment), "-c", verify_installation, MERIDIAN_URL, MERIDIAN_COMMIT, MERIDIAN_VERSION],
        cwd=checkout,
    )
    return versions.returncode == 0

candidate_envs = sorted(Path("/content").glob("tcc-meridian-*/venv"), key=lambda path: path.stat().st_mtime, reverse=True)
for candidate in candidate_envs:
    if (candidate / "bin" / "python").is_file() and verify_environment(candidate):
        venv_dir = candidate
        print("Ambiente da validação anterior reaproveitado.")
        break
else:
    subprocess.run([sys.executable, "-I", "-m", "venv", "--without-pip", str(venv_dir)], check=True)
    subprocess.run([*pip_command(venv_dir), "install", "--no-input", "-r", "requirements.txt"], cwd=checkout, check=True)
    if not verify_environment(venv_dir):
        raise RuntimeError("O ambiente instalado não passou nas verificações.")

python = python_command(venv_dir)
test_env = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": "-1",
    "JAX_PLATFORMS": "cpu",
    "MERIDIAN_BACKEND": "jax",
    "MPLCONFIGDIR": str(work_dir / "matplotlib"),
    "TF_CPP_MIN_LOG_LEVEL": "2",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
}
with (report_dir / "versoes_instaladas.txt").open("w", encoding="utf-8") as output:
    subprocess.run([*pip_command(venv_dir), "freeze", "--all"], stdout=output, cwd=checkout, check=True)
write_report("ambiente.json", {
    "commit_tcc": REPO_COMMIT, "python": platform.python_version(), "plataforma": platform.platform(),
    "backend": "jax", "dispositivo": "cpu", "meridian": MERIDIAN_VERSION, "commit_meridian": MERIDIAN_COMMIT,
})
requirements_verified = True


## 2. Conferir os dados e rodar os testes de integridade

In [ ]:
import xml.etree.ElementTree as ET

if not globals().get("requirements_verified", False):
    raise RuntimeError("Conclua a célula 1 primeiro.")

integrity_ok = False
run_dir = None
run_info = None
for previous_log in ("execucao.json", "execucao_eda.log"):
    (report_dir / previous_log).unlink(missing_ok=True)

expected_hashes = {
    "data/raw/geo_all_channels.csv": "d9ee016f7cd21f5c90b50da10a794af91a372b69f881d3caa266edd41fdafbf6",
    "data/processed/geo_time_panel.csv": "3aa1bc0ee293b7c6d46ccc5ad7c74583ec670a78153432be183d338ad11f4abc",
}
hashes = {
    name: {"esperado": expected, "observado": digest(checkout / name)}
    for name, expected in expected_hashes.items()
}
hashes_ok = all(item["esperado"] == item["observado"] for item in hashes.values())
junit_file = report_dir / "pytest_integridade.xml"
junit_file.unlink(missing_ok=True)
with (report_dir / "pytest_integridade.log").open("w", encoding="utf-8") as log:
    test_result = subprocess.run(
        [*python, "-m", "pytest", "-q", "-p", "no:cacheprovider",
         f"--junitxml={junit_file}", "tests/test_integrity.py"],
        cwd=checkout, env=test_env, stdout=log, stderr=subprocess.STDOUT,
    )

cases = ET.parse(junit_file).findall(".//testcase") if junit_file.is_file() else []
expected_intro_tests = {"test_migrated_hashes", "test_academic_preservation"}
old_intro_hash = "04bc9a7320cfa8f90c6053119198cf3483047d501c4eaf13cc9db3890166634e"
audited_intro_hash = "93a3faa368dc34e0a7d4af24f764efc095a1c4bb2fd71d5aa7e61d79030198cb"
known, unexpected, passed = [], [], []
for case in cases:
    name = case.attrib["name"]
    failure = case.find("failure")
    if failure is not None:
        detail = failure.text or ""
        if name in expected_intro_tests and old_intro_hash in detail and audited_intro_hash in detail:
            known.append(name)
        else:
            unexpected.append(name)
    elif case.find("error") is not None or case.find("skipped") is not None:
        unexpected.append(name)
    else:
        passed.append(name)

integrity_ok = (
    hashes_ok and len(cases) == 9 and not unexpected
    and set(known) == expected_intro_tests and len(passed) == 7
    and test_result.returncode == 1
)
write_report("integridade.json", {
    "apta_para_eda": integrity_ok, "dados": hashes, "pytest_codigo": test_result.returncode,
    "passaram": passed, "falhas_conhecidas": known, "falhas_inesperadas": unexpected,
})
print(f"Hashes da base e do painel conferidos: {hashes_ok}")
print(f"Pytest — {len(passed)} passaram, {len(known)} falhas conhecidas, {len(unexpected)} inesperadas.")
if not integrity_ok:
    raise RuntimeError("A integridade divergiu do esperado. Vá à célula 4 para baixar o diagnóstico.")


## 3. Executar a EDA completa na cópia isolada

In [ ]:
import time
from datetime import datetime, timezone

if not globals().get("integrity_ok", False):
    raise RuntimeError("Conclua a célula 2 antes de executar a EDA.")

runs_root = checkout / "validation_runs"
previous_runs = set(runs_root.glob("*"))
run_dir = None
run_info = None
command = [*python, "scripts/execute_notebook.py", "--mode", "full", "--engine", "inprocess"]
started = datetime.now(timezone.utc).isoformat()
started_ns = time.time_ns()
log_path = report_dir / "execucao_eda.log"
with log_path.open("w", encoding="utf-8", buffering=1) as log:
    with subprocess.Popen(
        command, cwd=checkout, env=test_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    ) as process:
        for line in process.stdout:
            log.write(line)
            if line.startswith(("Executando célula", "Diretório da execução:")):
                print(line, end="", flush=True)
        return_code = process.wait()

new_runs = [path for path in set(runs_root.glob("*")) - previous_runs if path.is_dir()]
if len(new_runs) == 1 and new_runs[0].resolve().parent == runs_root.resolve():
    run_dir = new_runs[0].resolve()
run_info = {
    "iniciada_utc": started, "inicio_ns": started_ns, "codigo_saida": return_code,
    "diretorio": str(run_dir) if run_dir else None, "argumentos": command[-4:],
}
write_report("execucao.json", run_info)
if return_code or run_dir is None:
    raise RuntimeError("O executor não concluiu a cópia isolada. Vá à célula 4 para baixar os logs.")
print("Execução concluída. Prossiga para comparar e baixar os resultados.")


## 4. Comparar os resultados e baixar o ZIP

Se o download não começar, abra o painel **Arquivos**, localize o caminho mostrado na saída e baixe **VALIDACAO_EDA_COLAB.zip**. Faça isso antes de encerrar a sessão.

In [ ]:
comparison_code = r'''
import json
import runpy
import sys
from hashlib import sha256
from pathlib import Path

import pandas as pd

reference, run_dir, report = map(Path, sys.argv[1:4])
started_ns = int(sys.argv[4])
commit, meridian_commit, raw_sha = sys.argv[5:]
eda_manifest_path = "outputs/logs/notebook_run_manifest.json"
temporal_manifest_path = "outputs/temporal_complement/logs/tc_execution_manifest.json"
meridian_metadata_path = "data/metadata/meridian_run.json"
html_path = "outputs/reports/meridian_eda_report.html"
issues = []

def digest(path):
    return sha256(path.read_bytes()).hexdigest()

def fresh(name):
    path = run_dir / name
    return path.is_file() and path.stat().st_size > 0 and path.stat().st_mtime_ns >= started_ns

def read_manifest(name):
    if not fresh(name):
        issues.append(f"Arquivo novo ausente: {name}")
        return {}
    return json.loads((run_dir / name).read_text(encoding="utf-8"))

stages = read_manifest(eda_manifest_path).get("stages", {})
temporal = read_manifest(temporal_manifest_path)
meridian = read_manifest(meridian_metadata_path)
if set(stages) != {f"{number:02}" for number in range(8)} or any(
    stage.get("status") != "complete" for stage in stages.values()
):
    issues.append("As etapas 00–07 não constam como concluídas no manifesto.")
if (temporal.get("rows"), temporal.get("geos"), temporal.get("weeks")) != (6240, 40, 156):
    issues.append("Dimensões da seção 08 divergentes.")
if temporal.get("git_commit_at_execution") != commit or temporal.get("source_dirty") is not False:
    issues.append("O manifesto temporal diverge do commit limpo esperado.")

raw = run_dir / "data/raw/geo_all_channels.csv"
panel = run_dir / "data/processed/geo_time_panel.csv"
if not raw.is_file() or digest(raw) != raw_sha or temporal.get("raw_sha256") != raw_sha:
    issues.append("SHA-256 da base bruta divergente na execução.")
if not panel.is_file() or temporal.get("processed_sha256") != digest(panel):
    issues.append("SHA-256 do painel divergente do manifesto temporal.")
if (meridian.get("version") != "2.1.0" or meridian.get("source_commit") != meridian_commit
        or meridian.get("posterior_sampled") is not False or "posterior" in meridian.get("inference_groups", [])):
    issues.append("O registro do Meridian não confirma o escopo exploratório e o commit esperado.")

recorded = {name: checksum for stage in stages.values() for name, checksum in stage.get("files", {}).items()}
temporal_files = temporal.get("files", {})
def evidence(name, files):
    if name not in files or not fresh(name):
        return "sem evidência de nova geração"
    if files[name] != digest(run_dir / name):
        return "hash divergente do manifesto"
    return "confirmada"

if evidence(html_path, recorded) != "confirmada":
    issues.append("O relatório HTML oficial não foi confirmado como novo.")
for path in (reference / "outputs/temporal_complement/figures").glob("*.png"):
    name = path.relative_to(reference).as_posix()
    if evidence(name, temporal_files) != "confirmada":
        issues.append(f"Figura temporal sem evidência válida: {name}")
for name in ("EDA_RESULTS.zip", "TEMPORAL_COMPLEMENT_RESULTS.zip", "executed.ipynb"):
    if not fresh(name):
        issues.append(f"Artefato novo ausente: {name}")

# Importa a função; a interface CLI gravaria sobre o relatório histórico.
compare_runs = runpy.run_path(str(reference / "scripts/compare_runs.py"))["compare"]
historical = compare_runs(run_dir)
for row in historical:
    name = "outputs/tables/" + row["file"]
    row["proveniencia"] = evidence(name, recorded)

temporal_rows = []
for path in sorted((reference / "outputs/temporal_complement/tables").glob("*.csv")):
    name = path.relative_to(reference).as_posix()
    new = run_dir / name
    row = {"file": path.name, "proveniencia": evidence(name, temporal_files)}
    if not new.is_file():
        row["status"] = "missing"
    else:
        try:
            left = pd.read_csv(path)
            right = pd.read_csv(new)
            pd.testing.assert_frame_equal(left, right, check_dtype=False, check_exact=False, rtol=1e-7, atol=1e-7)
            row.update(status="equivalent", rows=len(left))
        except Exception as error:
            row.update(status="different", detail=str(error)[:1800])
    temporal_rows.append(row)

def equivalents(rows):
    return sum(row["status"] == "equivalent" and row["proveniencia"] == "confirmada" for row in rows)

summary = {}
for key, rows, expected in (("historicas", historical, 129), ("temporais", temporal_rows, 15)):
    equal = equivalents(rows)
    summary[key] = {"comparadas": len(rows), "equivalentes_e_regeradas": equal, "esperadas": expected}
    if equal != expected or len(rows) != expected:
        issues.append(f"Tabelas {key}: {equal}/{expected} equivalentes e regeradas.")
result = {
    "validacao_aprovada": not issues, "rtol": 1e-7, "atol": 1e-7,
    "resumo": summary, "problemas": issues, "historicas": historical, "temporais": temporal_rows,
}
(report / "comparacao.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
'''

from zipfile import ZIP_DEFLATED, ZIP_STORED, ZipFile
from google.colab import files

if not globals().get("requirements_verified", False):
    raise RuntimeError("Conclua a célula 1 antes de exportar.")

run_dir = globals().get("run_dir")
run_info = globals().get("run_info")
comparison_file = report_dir / "comparacao.json"
comparison_file.unlink(missing_ok=True)
comparison_log = report_dir / "comparacao.log"
comparison_log.unlink(missing_ok=True)
comparison_ok = False
if run_dir is not None and run_info["codigo_saida"] == 0:
    with comparison_log.open("w", encoding="utf-8") as log:
        compare_result = subprocess.run(
            [*python, "-c", comparison_code, str(checkout), str(run_dir), str(report_dir),
             str(run_info["inicio_ns"]), REPO_COMMIT, MERIDIAN_COMMIT,
             expected_hashes["data/raw/geo_all_channels.csv"]],
            cwd=work_dir, env=test_env, stdout=log, stderr=subprocess.STDOUT,
        )

    comparison_ok = compare_result.returncode == 0

if comparison_ok and comparison_file.is_file():
    comparison_report = json.loads(comparison_file.read_text(encoding="utf-8"))
else:
    comparison_report = {"validacao_aprovada": False, "resumo": {}, "problemas": ["Comparação não concluída. Consulte os logs."]}

checkout_status = subprocess.check_output(
    [git, "status", "--porcelain", "--untracked-files=all"], cwd=checkout, text=True,
).strip()
if checkout_status:
    comparison_report["problemas"].append("A cópia de referência apresenta alterações.")
integrity_file = report_dir / "integridade.json"
integrity_report = json.loads(integrity_file.read_text(encoding="utf-8")) if integrity_file.is_file() else {}
success = comparison_report["validacao_aprovada"] and not checkout_status and integrity_report.get("apta_para_eda", False)
status = "EDA VALIDADA COM RESSALVA DOCUMENTADA" if success else "VALIDAÇÃO INCOMPLETA OU COM DIVERGÊNCIAS"
write_report("resumo_validacao.json", {
    "status": status, "commit_tcc": REPO_COMMIT, "checkout_alteracoes": checkout_status,
    "integridade": integrity_report, "execucao": run_info,
    "comparacao": comparison_report["resumo"], "problemas": comparison_report["problemas"],
})

artifact_names = (
    "EDA_RESULTS.zip", "TEMPORAL_COMPLEMENT_RESULTS.zip", "executed.ipynb",
    "outputs/logs/notebook_run_manifest.json", "outputs/temporal_complement/logs/tc_execution_manifest.json",
    "outputs/reports/meridian_eda_report.html", "data/metadata/meridian_run.json",
)
bundle = work_dir / "VALIDACAO_EDA_COLAB.zip"
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(report_dir.iterdir()):
        archive.write(file, f"relatorios/{file.name}")
    if run_dir is not None:
        for name in artifact_names:
            path = run_dir / name
            if path.is_file() and path.stat().st_mtime_ns >= run_info["inicio_ns"]:
                compression = ZIP_STORED if path.suffix == ".zip" else ZIP_DEFLATED
                archive.write(path, f"execucao/{name}", compress_type=compression)

print(status)
for group, counts in comparison_report["resumo"].items():
    print(f'{group}: {counts["equivalentes_e_regeradas"]}/{counts["esperadas"]} equivalentes e regeradas.')
for problem in comparison_report["problemas"]:
    print(problem)
print("Envie ao Codex", bundle)
files.download(str(bundle))
